<a href="https://colab.research.google.com/github/mdzikrim/MachineLearningClass/blob/main/Chapter_13_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Machine Learning Exercise 13

1. Why would you want to use the Data API?
> Untuk membuat pipeline input yang efisien dan scalable (streaming dari disk, transformasi, batching, prefetching, paralelisme), terutama saat dataset besar tidak muat RAM.

2. What are the benefits of splitting a large dataset into multiple files?
> Meningkatkan throughput I/O karena bisa dibaca paralel dan di-interleave, memudahkan sharding/distributed training, mengurangi bottleneck “satu file besar”, dan lebih fleksibel untuk penyimpanan/transfer. (Konsep pembacaan paralel & interleave disebut eksplisit di bagian TFRecordDataset).

3. During training, how can you tell that your input pipeline is the bottleneck?
What can you do to fix it?
> Indikasi umum: GPU/TPU utilization rendah sementara CPU/I/O sibuk; step training “menunggu data”. Perbaikan: paralelkan parsing/preprocessing (num_parallel_calls, num_parallel_reads), gunakan prefetch(), shard file (banyak TFRecord), optimalkan preprocessing, atau lakukan sebagian preprocessing lebih dulu (mis. TF Transform), dan bila perlu tingkatkan resource CPU/RAM/bandwidth.

4. Can you save any binary data to a TFRecord file, or only serialized protocol
buffers?
> Bisa menyimpan binary record arbitrer (apa pun), tetapi praktik umum: menyimpan protobuf ter-serialize.

5. Why would you go through the hassle of converting all your data to the Example
protobuf format? Why not use your own protobuf definition?
> Example protobuf didukung parser bawaan TensorFlow (tf.io.parse_*example()), fleksibel untuk banyak dataset. Custom protobuf memungkinkan format lebih spesifik, tetapi lebih rumit dan perlu descriptor saat deploy.

6. When using TFRecords, when would you want to activate compression? Why
not do it systematically?
> Aktifkan kompresi jika TFRecord harus diunduh lewat jaringan (file lebih kecil → download lebih cepat). Jika file lokal satu mesin, biasanya lebih baik tanpa kompresi untuk menghindari overhead CPU dekompresi.

7. Data can be preprocessed directly when writing the data files, or within the
tf.data pipeline, or in preprocessing layers within your model, or using TF Trans
form. Can you list a few pros and cons of each option?
> - Preprocess saat membuat file: training lebih cepat; bisa hemat storage; mudah inspeksi/arsip. Minus: sulit eksperimen banyak varian; augmentation memakan disk; aplikasi inferensi perlu preprocessing terpisah.
> - Preprocess di tf.data: mudah ubah logic & augmentation; bisa sangat efisien (multithread + prefetch). Minus: training lebih lambat; preprocessing terulang tiap epoch; model tetap “mengharapkan” input sudah terproses.
> - Preprocessing layers di model: satu kali tulis untuk training & inference; mengurangi risiko mismatch logic. Minus: training melambat; by default jalan di device batch (sering GPU) dan tidak otomatis mendapat paralelisme CPU/prefetch (walau roadmap Keras bisa mengangkatnya ke pipeline).
> - TF Transform: materialize data terproses (sekali), training cepat; preprocessing layer dihasilkan otomatis. Minus: perlu belajar tool tambahan.

8. Name a few common techniques you can use to encode categorical features.
What about text?
> - Kategorikal ber-urutan: ordinal encoding (urut alami → rank).
> - Kategorikal tanpa urutan: one-hot; jika kategori banyak: embeddings.
> - Teks: bag-of-words (sering pakai TF-IDF), bisa juga n-grams, atau word embeddings (pretrained/learned). Juga opsi token karakter/subword (dibahas di ch.16).

9. Load the Fashion MNIST dataset (introduced in Chapter 10); split it into a train
ing set, a validation set, and a test set; shuffle the training set; and save each
dataset to multiple TFRecord files. Each record should be a serialized Example
protobuf with two features: the serialized image (use tf.io.serialize_tensor()
 to serialize each image), and the label.11 Then use tf.data to create an efficient
dataset for each set. Finally, use a Keras model to train these datasets, including a
preprocessing layer to standardize each input feature. Try to make the input
pipeline as efficient as possible, using TensorBoard to visualize profiling data.

In [1]:
import os, math, time
import numpy as np
import tensorflow as tf
from tensorflow import keras

AUTOTUNE = tf.data.AUTOTUNE

In [2]:
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# split: 55k train, 5k valid (silakan ubah jika Anda mau)
X_valid, y_valid = X_train_full[:5000], y_train_full[:5000]
X_train, y_train = X_train_full[5000:], y_train_full[5000:]

# add channel dim: (28,28,1)
X_train = X_train[..., np.newaxis]
X_valid = X_valid[..., np.newaxis]
X_test  = X_test[...,  np.newaxis]

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
def _bytes_feature(value: bytes) -> tf.train.Feature:
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _int64_feature(value: int) -> tf.train.Feature:
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[int(value)]))

def serialize_example(image_uint8, label_int):
    # image_uint8: uint8 tensor/array shape (28,28,1)
    image_tensor = tf.convert_to_tensor(image_uint8, dtype=tf.uint8)
    image_bytes = tf.io.serialize_tensor(image_tensor)  # required by the exercise
    feature = {
        "image": _bytes_feature(image_bytes.numpy()),
        "label": _int64_feature(label_int),
    }
    example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
    return example_proto.SerializeToString()

def write_tfrecord_shards(X, y, out_dir, prefix, n_shards=10):
    os.makedirs(out_dir, exist_ok=True)
    n = len(X)
    shard_size = math.ceil(n / n_shards)
    paths = []
    for shard_id in range(n_shards):
        start = shard_id * shard_size
        end = min(start + shard_size, n)
        if start >= end:
            break
        path = os.path.join(out_dir, f"{prefix}_{shard_id:02d}.tfrecord")
        paths.append(path)
        with tf.io.TFRecordWriter(path) as f:
            for img, lbl in zip(X[start:end], y[start:end]):
                f.write(serialize_example(img, lbl))
    return paths

DATA_DIR = "tfrec_fashion_mnist"
train_paths = write_tfrecord_shards(X_train, y_train, DATA_DIR, "train", n_shards=20)
valid_paths = write_tfrecord_shards(X_valid, y_valid, DATA_DIR, "valid", n_shards=5)
test_paths  = write_tfrecord_shards(X_test,  y_test,  DATA_DIR, "test",  n_shards=5)

In [4]:
feature_description = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
}

def parse_example(serialized):
    ex = tf.io.parse_single_example(serialized, feature_description)
    image = tf.io.parse_tensor(ex["image"], out_type=tf.uint8)
    image = tf.ensure_shape(image, [28, 28, 1])
    image = tf.cast(image, tf.float32)  # will standardize later
    label = tf.cast(ex["label"], tf.int32)
    return image, label

def make_dataset(filepaths, training=False, batch_size=32, cache=False):
    # reading multiple files in parallel:
    ds = tf.data.TFRecordDataset(filepaths, num_parallel_reads=AUTOTUNE)
    ds = ds.map(parse_example, num_parallel_calls=AUTOTUNE)

    if cache:
        ds = ds.cache()

    if training:
        ds = ds.shuffle(10_000, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds

batch_size = 64
train_ds = make_dataset(train_paths, training=True,  batch_size=batch_size, cache=False)
valid_ds = make_dataset(valid_paths, training=False, batch_size=batch_size, cache=False)
test_ds  = make_dataset(test_paths,  training=False, batch_size=batch_size, cache=False)

In [5]:
norm = keras.layers.Normalization(axis=(1, 2, 3))
norm.adapt(train_ds.map(lambda x, y: x).take(200))

In [6]:
model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    norm,
    keras.layers.Conv2D(32, 3, activation="relu"),
    keras.layers.MaxPool2D(),
    keras.layers.Conv2D(64, 3, activation="relu"),
    keras.layers.MaxPool2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

log_dir = os.path.join("logs", "ch13_ex9", time.strftime("%Y%m%d-%H%M%S"))
tb = keras.callbacks.TensorBoard(
    log_dir=log_dir,
    profile_batch=(100, 120)
)

history = model.fit(train_ds, validation_data=valid_ds, epochs=5, callbacks=[tb])
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)

print("TensorBoard logdir:", log_dir)

Epoch 1/5
    860/Unknown 58s 64ms/step - accuracy: 0.7798 - loss: 0.6172

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


860/860 ━━━━━━━━━━━━━━━━━━━━ 59s 66ms/step - accuracy: 0.7799 - loss: 0.6170 - val_accuracy: 0.8774 - val_loss: 0.3398
Epoch 2/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 82s 95ms/step - accuracy: 0.8847 - loss: 0.3133 - val_accuracy: 0.8884 - val_loss: 0.2959
Epoch 3/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 51s 58ms/step - accuracy: 0.9021 - loss: 0.2624 - val_accuracy: 0.9044 - val_loss: 0.2704
Epoch 4/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 48s 55ms/step - accuracy: 0.9165 - loss: 0.2241 - val_accuracy: 0.9044 - val_loss: 0.2673
Epoch 5/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 48s 55ms/step - accuracy: 0.9263 - loss: 0.1976 - val_accuracy: 0.9140 - val_loss: 0.2456
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9068 - loss: 0.2677
Test accuracy: 0.9071999788284302
TensorBoard logdir: logs/ch13_ex9/20260110-142057


10. In this exercise you will download a dataset, split it, create a tf.data.Dataset to
load it and preprocess it efficiently, then build and train a binary classification
model containing an Embedding layer:

In [7]:
import os, re, random, tarfile
import numpy as np
import tensorflow as tf
from tensorflow import keras

AUTOTUNE = tf.data.AUTOTUNE

In [9]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
data_dir = keras.utils.get_file("aclImdb_v1.tar.gz", url, extract=True)
# keras.utils.get_file returns path to archive; extracted dir is sibling:
# Corrected: data_dir points to 'aclImdb_v1_extracted', and 'aclImdb' is inside it.
acl_root = os.path.join(data_dir, "aclImdb")

train_dir = os.path.join(acl_root, "train")
test_dir  = os.path.join(acl_root, "test")

def list_imdb_files(split_dir):
    filepaths = []
    labels = []
    for label_name, label in [("neg", 0), ("pos", 1)]:
        dirpath = os.path.join(split_dir, label_name)
        for fname in os.listdir(dirpath):
            if fname.endswith(".txt"):
                filepaths.append(os.path.join(dirpath, fname))
                labels.append(label)
    return np.array(filepaths), np.array(labels, dtype=np.int32)

train_files, train_labels = list_imdb_files(train_dir)
test_files,  test_labels  = list_imdb_files(test_dir)

# Shuffle train
rng = np.random.default_rng(42)
idx = rng.permutation(len(train_files))
train_files, train_labels = train_files[idx], train_labels[idx]

In [10]:
idx = rng.permutation(len(test_files))
test_files, test_labels = test_files[idx], test_labels[idx]

valid_size = 15_000
valid_files, valid_labels = test_files[:valid_size], test_labels[:valid_size]
final_test_files, final_test_labels = test_files[valid_size:valid_size + 10_000], test_labels[valid_size:valid_size + 10_000]

In [11]:
def make_text_dataset(filepaths, labels, training=False, batch_size=32, cache=False):
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if training:
        ds = ds.shuffle(20_000, reshuffle_each_iteration=True)

    def _load_text(path, y):
        text = tf.io.read_file(path)
        return text, y

    ds = ds.map(_load_text, num_parallel_calls=AUTOTUNE)

    if cache:
        ds = ds.cache()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds

batch_size = 64
train_ds = make_text_dataset(train_files, train_labels, training=True,  batch_size=batch_size, cache=False)
valid_ds = make_text_dataset(valid_files, valid_labels, training=False, batch_size=batch_size, cache=False)
test_ds  = make_text_dataset(final_test_files, final_test_labels, training=False, batch_size=batch_size, cache=False)

In [12]:
max_tokens = 20_000
seq_len = 250

vectorize = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=seq_len,
)

# adapt on training text only
text_only = train_ds.map(lambda text, y: text)
vectorize.adapt(text_only)

In [13]:
embed_dim = 128

class RescaledMeanEmbedding(keras.layers.Layer):
    def __init__(self, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embedding = keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)

    def call(self, token_ids):
        # token_ids: (batch, time)
        emb = self.embedding(token_ids)  # (batch, time, dim)

        # mask: 1 for real tokens, 0 for padding
        mask = tf.cast(token_ids != 0, tf.float32)  # (batch, time)
        lengths = tf.reduce_sum(mask, axis=1, keepdims=True)  # (batch, 1)

        # sum embeddings with mask
        mask3 = mask[..., tf.newaxis]  # (batch, time, 1)
        summed = tf.reduce_sum(emb * mask3, axis=1)  # (batch, dim)

        mean = summed / tf.maximum(lengths, 1.0)  # avoid div0
        rescaled = mean * tf.sqrt(tf.maximum(lengths, 1.0))
        return rescaled

inputs = keras.layers.Input(shape=(), dtype=tf.string)
x = vectorize(inputs)
x = RescaledMeanEmbedding(max_tokens, embed_dim)(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [14]:
history = model.fit(train_ds, validation_data=valid_ds, epochs=5)
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 21s 49ms/step - accuracy: 0.7878 - loss: 0.4510 - val_accuracy: 0.8651 - val_loss: 0.3164
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accuracy: 0.9363 - loss: 0.1747 - val_accuracy: 0.8581 - val_loss: 0.3410
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 46ms/step - accuracy: 0.9624 - loss: 0.1063 - val_accuracy: 0.8458 - val_loss: 0.3937
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 50ms/step - accuracy: 0.9752 - loss: 0.0685 - val_accuracy: 0.8319 - val_loss: 0.5998
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.9824 - loss: 0.0477 - val_accuracy: 0.8402 - val_loss: 0.6438
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8430 - loss: 0.6354
Test accuracy: 0.840399980545044


In [15]:
import tensorflow_datasets as tfds

datasets, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True)
raw_train = datasets["train"]
raw_test  = datasets["test"]

# split TFDS test: 15k valid, 10k test (TFDS test has 25k)
raw_test = raw_test.shuffle(25_000, seed=42, reshuffle_each_iteration=False)
raw_valid = raw_test.take(15_000)
raw_final_test = raw_test.skip(15_000).take(10_000)

# batch/prefetch
raw_train_b = raw_train.shuffle(25_000, seed=42).batch(batch_size).prefetch(AUTOTUNE)
raw_valid_b = raw_valid.batch(batch_size).prefetch(AUTOTUNE)
raw_test_b  = raw_final_test.batch(batch_size).prefetch(AUTOTUNE)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.HXU4E1_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.HXU4E1_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.HXU4E1_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.
